# Lab: Cache-Aware Routing & Semantic Caching

This lab explores how intelligent routing and caching strategies reduce redundant KV cache computation in LLM serving systems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import hashlib
from typing import Dict, List, Tuple

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## Simulating Prefix Cache Hit Rates

We compare round-robin routing (ignores cache state) vs prefix-aware routing (sends requests to the node that already has the matching prefix cached).

In [ ]:
# Simulation parameters
NUM_REQUESTS = 1000
NUM_NODES = 4
NUM_TEMPLATES = 5
PROMPT_TEMPLATES = [f'system_prompt_template_{i}' for i in range(NUM_TEMPLATES)]

# Generate requests: each has a system prompt prefix + unique user query
requests = [(np.random.choice(PROMPT_TEMPLATES), f'user_query_{i}') for i in range(NUM_REQUESTS)]

# Round-robin routing
def round_robin_route(requests, num_nodes):
    node_caches = defaultdict(set)  # node -> set of cached prefixes
    hits, misses = 0, 0
    for i, (prefix, _) in enumerate(requests):
        node = i % num_nodes
        if prefix in node_caches[node]:
            hits += 1
        else:
            misses += 1
            node_caches[node].add(prefix)
    return hits / len(requests)

# Prefix-aware routing: hash prefix to consistent node
def prefix_aware_route(requests, num_nodes):
    node_caches = defaultdict(set)
    hits, misses = 0, 0
    for prefix, _ in requests:
        node = int(hashlib.md5(prefix.encode()).hexdigest(), 16) % num_nodes
        if prefix in node_caches[node]:
            hits += 1
        else:
            misses += 1
            node_caches[node].add(prefix)
    return hits / len(requests)

rr_hit_rate = round_robin_route(requests, NUM_NODES)
pa_hit_rate = prefix_aware_route(requests, NUM_NODES)

print(f'Round-Robin Cache Hit Rate: {rr_hit_rate:.3f}')
print(f'Prefix-Aware Cache Hit Rate: {pa_hit_rate:.3f}')
print(f'Improvement: {(pa_hit_rate - rr_hit_rate) / max(rr_hit_rate, 0.001) * 100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
strategies = ['Round-Robin', 'Prefix-Aware']
hit_rates = [rr_hit_rate, pa_hit_rate]
colors = ['#ef4444', '#22c55e']

bars = ax.bar(strategies, hit_rates, color=colors, edgecolor='black', width=0.5)
ax.set_ylabel('Cache Hit Rate')
ax.set_title('Prefix Cache Hit Rate by Routing Strategy')
ax.set_ylim(0, 1.05)
for bar, rate in zip(bars, hit_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{rate:.1%}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('routing_hit_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## Session Affinity: Multi-Turn Savings

In multi-turn conversations, session affinity pins a conversation to the same node, allowing KV cache reuse across turns. Without it, each turn recomputes the full context from scratch.

In [ ]:
# Simulate multi-turn conversations
NUM_CONVERSATIONS = 50
TURNS_PER_CONVERSATION = 10
TOKENS_PER_TURN = 200  # new tokens per turn
PREFILL_COST_PER_TOKEN_MS = 0.05  # ms per token for prefill

def simulate_sessions(use_affinity: bool, num_nodes: int = 4):
    """Returns cumulative prefill tokens saved at each turn."""
    total_prefill_tokens = []  # tokens that must be prefilled per turn
    cumulative = 0
    for conv in range(NUM_CONVERSATIONS):
        affinity_node = np.random.randint(num_nodes)
        cached_tokens = 0
        for turn in range(TURNS_PER_CONVERSATION):
            context_length = (turn + 1) * TOKENS_PER_TURN
            if use_affinity:
                # Same node: only prefill new tokens
                prefill_needed = TOKENS_PER_TURN
            else:
                # Random node: full context prefill each turn
                prefill_needed = context_length
            cumulative += prefill_needed
            total_prefill_tokens.append(cumulative)
    return total_prefill_tokens

no_affinity = simulate_sessions(use_affinity=False)
with_affinity = simulate_sessions(use_affinity=True)

savings_pct = (1 - with_affinity[-1] / no_affinity[-1]) * 100
print(f'Without affinity: {no_affinity[-1]:,} total prefill tokens')
print(f'With affinity:    {with_affinity[-1]:,} total prefill tokens')
print(f'Savings:          {savings_pct:.1f}% fewer prefill tokens')

# Plot cumulative savings
fig, ax = plt.subplots(figsize=(10, 5))
turns = range(len(no_affinity))
ax.plot(turns, np.array(no_affinity) / 1e6, label='No Affinity', color='#ef4444', linewidth=2)
ax.plot(turns, np.array(with_affinity) / 1e6, label='With Session Affinity', color='#22c55e', linewidth=2)
ax.fill_between(turns, np.array(with_affinity)/1e6, np.array(no_affinity)/1e6, alpha=0.15, color='green')
ax.set_xlabel('Request Index (50 conversations x 10 turns)')
ax.set_ylabel('Cumulative Prefill Tokens (millions)')
ax.set_title('KV Cache Reuse: Session Affinity vs Random Routing')
ax.legend()
plt.tight_layout()
plt.savefig('session_affinity_savings.png', dpi=150, bbox_inches='tight')
plt.show()

## Semantic Cache with Cosine Similarity

A semantic cache stores embeddings of past queries. New queries are matched against the cache using cosine similarity. If a match exceeds the threshold, the cached response is returned (avoiding a full LLM call).

In [ ]:
# Simulate semantic cache with random embeddings
EMBED_DIM = 128
CACHE_SIZE = 100
NUM_QUERIES = 500
PARAPHRASE_NOISE = 0.1  # lower = more similar paraphrase

# Build cache: 100 canonical query embeddings
cache_embeddings = np.random.randn(CACHE_SIZE, EMBED_DIM)
cache_embeddings /= np.linalg.norm(cache_embeddings, axis=1, keepdims=True)

# Generate test queries: 50% paraphrases (should hit), 50% novel (should miss)
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def evaluate_cache(threshold: float):
    hits, false_positives, true_negatives, true_positives = 0, 0, 0, 0
    for i in range(NUM_QUERIES):
        is_paraphrase = i < NUM_QUERIES // 2
        if is_paraphrase:
            # Create a noisy version of a cached query
            orig = cache_embeddings[i % CACHE_SIZE]
            noise = np.random.randn(EMBED_DIM) * PARAPHRASE_NOISE
            query = orig + noise
            query /= np.linalg.norm(query)
        else:
            # Truly novel query
            query = np.random.randn(EMBED_DIM)
            query /= np.linalg.norm(query)
        
        # Find best match in cache
        sims = cache_embeddings @ query
        best_sim = sims.max()
        
        if best_sim >= threshold:
            hits += 1
            if is_paraphrase:
                true_positives += 1
            else:
                false_positives += 1
        else:
            if not is_paraphrase:
                true_negatives += 1
    
    hit_rate = hits / NUM_QUERIES
    fp_rate = false_positives / (NUM_QUERIES // 2) if (NUM_QUERIES // 2) > 0 else 0
    return hit_rate, fp_rate

thresholds = np.arange(0.5, 1.0, 0.05)
results = [evaluate_cache(t) for t in thresholds]
hit_rates = [r[0] for r in results]
fp_rates = [r[1] for r in results]

print(f'{"Threshold":<12} {"Hit Rate":<12} {"FP Rate":<12}')
print('-' * 36)
for t, hr, fp in zip(thresholds, hit_rates, fp_rates):
    print(f'{t:<12.2f} {hr:<12.3f} {fp:<12.3f}')

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

l1 = ax1.plot(thresholds, hit_rates, 'o-', color='#2563eb', linewidth=2, label='Hit Rate')
l2 = ax2.plot(thresholds, fp_rates, 's--', color='#dc2626', linewidth=2, label='False Positive Rate')

ax1.set_xlabel('Cosine Similarity Threshold')
ax1.set_ylabel('Cache Hit Rate', color='#2563eb')
ax2.set_ylabel('False Positive Rate', color='#dc2626')
ax1.set_title('Semantic Cache: Hit Rate vs False Positive Rate')
ax1.set_ylim(0, 1)
ax2.set_ylim(0, 1)

lines = l1 + l2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right')
plt.tight_layout()
plt.savefig('semantic_cache_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

## Cost Impact Analysis

We estimate monthly cost savings from prefix caching and semantic caching at different traffic levels, assuming $0.01 per 1K input tokens (prefill) as the baseline cost.

In [ ]:
# Cost model
COST_PER_1K_TOKENS = 0.01  # $/1K input tokens
AVG_PREFIX_TOKENS = 500    # average reusable prefix length
CACHE_HIT_RATE = 0.80      # achievable with prefix-aware routing
SEMANTIC_HIT_RATE = 0.30   # semantic cache avoids full LLM call
AVG_TOTAL_TOKENS = 1500    # average full request tokens

traffic_levels = [1_000, 10_000, 100_000]  # requests per day
labels = ['1K req/day', '10K req/day', '100K req/day']

print(f'{"Traffic":<14} {"Baseline/mo":<14} {"Prefix Save":<14} {"Semantic Save":<14} {"Total Save":<14}')
print('-' * 70)

savings_data = []
for daily_req, label in zip(traffic_levels, labels):
    monthly_req = daily_req * 30
    baseline_cost = monthly_req * AVG_TOTAL_TOKENS / 1000 * COST_PER_1K_TOKENS
    
    # Prefix caching saves prefill cost for cached prefixes
    prefix_savings = monthly_req * AVG_PREFIX_TOKENS / 1000 * COST_PER_1K_TOKENS * CACHE_HIT_RATE
    
    # Semantic cache avoids entire request cost for cache hits
    semantic_savings = monthly_req * SEMANTIC_HIT_RATE * AVG_TOTAL_TOKENS / 1000 * COST_PER_1K_TOKENS
    
    total_savings = prefix_savings + semantic_savings
    savings_data.append((baseline_cost, prefix_savings, semantic_savings, total_savings))
    print(f'{label:<14} ${baseline_cost:<13,.0f} ${prefix_savings:<13,.0f} ${semantic_savings:<13,.0f} ${total_savings:<13,.0f}')

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(labels))
w = 0.25
ax.bar(x - w, [s[1] for s in savings_data], w, label='Prefix Cache Savings', color='#22c55e', edgecolor='black')
ax.bar(x, [s[2] for s in savings_data], w, label='Semantic Cache Savings', color='#3b82f6', edgecolor='black')
ax.bar(x + w, [s[3] for s in savings_data], w, label='Total Savings', color='#f59e0b', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Monthly Savings ($)')
ax.set_title('Monthly Cost Savings from Cache-Aware Routing')
ax.legend()
plt.tight_layout()
plt.savefig('cost_savings.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **Prefix-aware routing** achieves near-perfect cache hit rates by consistently mapping the same system prompt prefix to the same node (vs ~75-80% with round-robin)
2. **Session affinity** dramatically reduces prefill computation in multi-turn conversations by reusing KV cache across turns (60-80% prefill savings)
3. **Semantic caching** trades off hit rate vs false positive rate: threshold ~0.85-0.90 balances correctness with cost savings
4. **Cost impact scales linearly** with traffic: at 100K req/day, combined caching can save thousands per month
5. **Production consideration**: Semantic cache invalidation and staleness detection are critical for correctness